# VLA 연구 · 12회차 — VLM → VLA 실행 파이프라인

| 항목 | 내용 |
|---|---|
| 오늘 목표 | `이미지 + 명령` → **VLM이 순차 지시로 분해** → **VLA가 관절 제어로 실행** 을 끝까지 돌린다 |
| 환경 | MuJoCo + Franka Panda, 관측·액션은 **LIBERO 호환** 규격 |
| 산출물 | 파이프라인 rollout GIF, VLM 계획 채점표, 텔레옵 데이터셋, SmolVLA 학습 준비 |

지금까지와 달라진 점: 10·11회차의 `MiniVLAController` 는 물체 좌표를 목표까지 **보간**해서
옮기는 흉내였습니다. 이번에는 **실제로 그리퍼로 집습니다** (물리 접촉·마찰 기반).


---
## SECTION 0 | 설치와 코드 연결

`vla_pipeline` 폴더가 팀 공유 드라이브에 있어야 합니다.

```
MyDrive/VLA_Team_2026/
└── vla_pipeline/          <- make_student_zip.py 로 만든 zip 을 푼 것
    ├── vlapipe/  scripts/  assets/  notebooks/
    └── README.md ...
```

**공유 폴더에서 바로 돌리지 않습니다.** 아래 0-2 가 코드를 런타임 로컬(`/content`)로
복사한 뒤 거기서 작업합니다. 이유는 셀 주석에 적어 두었습니다.
결과물은 마지막 셀에서 다시 공유 드라이브로 올립니다.


In [1]:
# 0-1. 패키지 설치 (Colab 최초 1회, 이후 런타임 재시작)
!pip install -q 'mujoco>=3.2,<4.0' h5py imageio imageio-ffmpeg 'transformers>=4.50' accelerate num2words torchvision
print('설치 완료 — Runtime > Restart session 후 0-2 부터 실행하세요.')

In [ ]:
# 0-2. Drive 마운트 + 코드를 런타임 로컬로 복사
import os, sys, shutil
os.environ['MUJOCO_GL'] = 'egl'
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    TEAM = Path('/content/drive/MyDrive/VLA_Team_2026')
    SRC = TEAM / 'vla_pipeline'
    assert SRC.exists(), f'공유 드라이브에 없습니다: {SRC}'

    # 공유 폴더에서 바로 돌리면 안 되는 이유:
    #  1) SortingEnv 는 만들 때마다 assets/.../vla_scene.xml 을 새로 씁니다.
    #     셋이 동시에 돌리면 서로의 씬 파일을 덮어써 ParseXML 오류가 납니다.
    #  2) 학습 체크포인트(outputs/smolvla_sorting)가 GB 단위로 쌓여
    #     공유 드라이브 용량을 먹습니다.
    #  3) Drive(FUSE)는 느려서 메시 파일 로딩이 오래 걸립니다.
    # -> 코드와 에셋만 런타임 로컬로 복사해 거기서 작업합니다. 원본은 건드리지 않습니다.
    PKG = Path('/content/vla_pipeline')
    shutil.rmtree(PKG / 'data', ignore_errors=True)      # 새로 올린 녹화본을 반영
    shutil.copytree(SRC, PKG, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('.venv', 'outputs', '__pycache__', '.git'))
    print('코드 사본 :', PKG)
    print('원본      :', SRC, '(읽기만 함)')
else:
    PKG = Path.cwd().parent
    TEAM = PKG.parent

assert PKG.exists(), f'경로를 찾을 수 없습니다: {PKG}'
sys.path.insert(0, str(PKG))
os.chdir(PKG)
(PKG / 'outputs').mkdir(exist_ok=True)

n_ep = len(list((PKG / 'data' / 'episodes').glob('*.hdf5'))) if (PKG / 'data' / 'episodes').exists() else 0
print('작업 폴더 :', PKG)
print('녹화본    :', n_ep, '개')


In [3]:
# 0-3. Panda 에셋 준비 (최초 1회, 약 40MB)
!python setup_assets.py

In [4]:
# 0-4. 환경 점검
!python scripts/check_env.py

---
## SECTION 1 | 씬 살펴보기

260827 문서에서 확정한 규격 그대로입니다: 작업 영역 0.8 × 0.5 m, 3×1 구역, 물체 10개, 카메라 3대 + 손목.

**정돈된 상태** = 긴 블록 5개는 오른쪽 보관 구역, 정육면체·타일 5개는 왼쪽 보관 구역, 가운데는 비어 있음.


In [5]:
import numpy as np, matplotlib.pyplot as plt
from vlapipe.env import SortingEnv
from vlapipe import config as C

env = SortingEnv(layout='tidy', image_size=256, cameras=C.ALL_CAMERAS)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for r, (lay, seed) in enumerate([('tidy', 0), ('messy', 3)]):
    env.reset(layout=lay, seed=seed)
    for c, cam in enumerate(C.ALL_CAMERAS):
        axes[r, c].imshow(env.render(cam, 300, 300)); axes[r, c].axis('off')
        axes[r, c].set_title(f'{lay} / {cam}', fontsize=9)
plt.tight_layout(); plt.show()
print('잘못 놓인 물체:', env.misplaced_objects())

### 왜 `agentview` 는 위에서 내려다보나요?

문서의 `frontview`(로봇을 마주보는 면)는 **영상의 좌우가 월드 좌우와 뒤집힙니다.**
그 시점으로 학습하면 VLM 이 말하는 "왼쪽"과 로봇 기준 "왼쪽"이 반대가 됩니다.
그래서 학습·추론용 주 카메라는 top-down 으로 두었습니다 (영상 오른쪽 = 월드 +x).

> **생각해 볼 것**: 실제 로봇에 옮길 때 천장 카메라를 달 수 없다면 어떻게 해야 할까요?


---
## SECTION 2 | 액션과 관측 규격 (LIBERO 호환)

```
action = [dx, dy, dz, drx, dry, drz, gripper]      # 7차원
  앞 3개 : 위치 델타, [-1,1] -> 최대 ±0.05 m
  중간 3개: 회전 델타(축각), [-1,1] -> 최대 ±0.5 rad
  마지막 : 그리퍼, -1 = 열기 / +1 = 닫기
```
이 규격을 LIBERO 와 맞춰 두면 나중에 LIBERO 사전학습 체크포인트나 데이터셋을 그대로 붙일 수 있습니다.


In [6]:
obs = env.reset(layout='messy', seed=3)
for k, v in obs.items():
    print(f'  {k:28s} {np.asarray(v).shape} {np.asarray(v).dtype}')
print('\nstate 벡터:', SortingEnv.state_vector(obs).shape)
print('제어 주기:', C.CONTROL_HZ, 'Hz | 액션당 시뮬 스텝:', C.SIM_STEPS_PER_ACTION)

In [7]:
# 액션을 직접 넣어 보기 — 그리퍼를 앞(+y)으로 밀고 내린다
env.reset(layout='messy', seed=3)
print('시작 eef:', env.ctrl.eef_pos().round(4))
for _ in range(6):
    env.step([0.0, 0.6, -0.5, 0, 0, 0, -1.0])
print('이후 eef:', env.ctrl.eef_pos().round(4))
plt.figure(figsize=(4,4)); plt.imshow(env.render('agentview', 300, 300)); plt.axis('off'); plt.show()

---
## SECTION 3 | 1단계 — VLM 플래너

`RulePlanner` 는 시뮬레이터 상태를 직접 보고 만든 **정답 계획**입니다. VLM 채점 기준으로 씁니다.


In [8]:
from vlapipe.planner import RulePlanner, VLMPlanner, plan_accuracy, parse_plan

env.reset(layout='messy', seed=3)
gold = RulePlanner(env).plan('테이블을 정리하시오')
for s in gold:
    print(f"  {s['step']}. {s['instruction']}")
    print(f"     {s['instruction_ko']}   -> ({s['object']}, {s['target_zone']})")

In [9]:
# SmolVLM2 로 같은 장면을 계획하게 한다 (최초 실행 시 모델 다운로드)
img = env.render('agentview', 448, 448)
vlm = VLMPlanner('HuggingFaceTB/SmolVLM2-500M-Video-Instruct')  # 기본 prompt_style='json'
raw = vlm.raw_response(img, '테이블을 정리하시오')
print('=== VLM 원문 ==='); print(raw)
pred = parse_plan(raw)
print('\n=== 파싱된 계획 ==='); [print('  ', s['step'], s['object'], '->', s['target_zone']) for s in pred]
print('\n=== 채점 ==='); print(plan_accuracy(pred, gold))

### ✏️ 미니과제 — 프롬프트 배틀 (7회차의 확장판)

조교가 미리 재 본 값입니다 (씬 3개, RTX 4080):

| 모델 | 프롬프트 | recall | 추론 |
|---|---|---|---|
| SmolVLM2-500M | `json` | 0.23 | 1.5 s |
| SmolVLM2-500M | `lines` | 0.00 | 0.5 s |
| SmolVLM2-2.2B | `json` | ~0.31 | **55 s** |

여기서 이미 세 가지가 보입니다.

1. **2.2B 는 값어치를 못 합니다** — 정확도는 조금 오르는데 추론이 40배 느립니다.
2. **2.2B 는 이미지를 거의 안 봅니다** — 씬을 바꿔도 출력 앞부분이 거의 똑같습니다.
   프롬프트에 쓴 규칙을 베껴 쓰는 것에 가깝습니다.
3. **`lines` 는 실패했습니다** — "한 줄에 하나씩"이 작은 모델에 쉬울 것 같지만,
   실제로는 한 줄만 쓰고 멈춥니다. 추측이 실측으로 뒤집힌 사례입니다.

**여러분이 할 일**: `vlapipe/planner.py` 의 `_PROMPT` 를 고쳐 이 표를 갱신하세요.
팀원 3명이 서로 다른 전략으로 붙어 봅니다.

- 구역을 색으로 부를까(초록/파랑), 위치로 부를까(왼쪽/오른쪽)?
- 물체 목록을 프롬프트에 미리 알려 줄까?
- 이미지 해상도(256 / 384 / 448)를 바꾸면 달라지나?
- "가운데에 무엇이 있나?" 만 먼저 묻고 목적지는 규칙으로 정하면?

> **이 결과가 보고서에 들어갑니다.** 3번 관찰(모델이 이미지를 안 본다)은
> "왜 VLM 을 파인튜닝해야 하는가"의 직접적인 근거입니다.


In [ ]:
# 여러 씬에서 VLM 계획 정확도를 재는 채점 루프
import pandas as pd
rows = []
for seed in [3, 7, 11, 19, 23]:
    env.reset(layout='messy', seed=seed)
    g = RulePlanner(env).plan()
    p = parse_plan(vlm.raw_response(env.render('agentview', 448, 448), '테이블을 정리하시오'))
    a = plan_accuracy(p, g)
    rows.append({'seed': seed, '정답수': a['n_gold'], '예측수': a['n_pred'],
                 '맞춘수': a['correct'], 'recall': round(a['recall'], 2),
                 'precision': round(a['precision'], 2)})
df = pd.DataFrame(rows); display(df)
print('평균 recall %.2f  precision %.2f' % (df.recall.mean(), df.precision.mean()))

In [ ]:
# 프롬프트 A/B 비교 — 같은 씬에 두 프롬프트를 붙여 본다
from vlapipe.planner import PROMPTS

def score_prompt(style, seeds=(3, 7, 11), img_size=448):
    vp = VLMPlanner('HuggingFaceTB/SmolVLM2-500M-Video-Instruct', prompt_style=style)
    c = g = n = 0
    for sd in seeds:
        env.reset(layout='messy', seed=sd)
        gold = RulePlanner(env).plan()
        pred = parse_plan(vp.raw_response(env.render('agentview', img_size, img_size), '테이블을 정리하시오'))
        a = plan_accuracy(pred, gold)
        c += a['correct']; g += a['n_gold']; n += a['n_pred']
    del vp
    import torch, gc; gc.collect(); torch.cuda.empty_cache()
    return {'프롬프트': style, 'recall': round(c / g, 2),
            'precision': round(c / n, 2) if n else 0.0, '맞춘수': c, '정답수': g, '예측수': n}

display(pd.DataFrame([score_prompt(s) for s in PROMPTS]))

---
## SECTION 4 | 2단계 — VLA 정책

`ScriptedSortingPolicy` 는 **teacher(선생) 정책**입니다. 물체 위치를 시뮬레이터에서 직접 읽으므로
학습된 정책이 아닙니다. 두 가지 용도로 씁니다:

1. SmolVLA 를 학습시키기 전에도 파이프라인이 끝까지 돌게 하는 **대역**
2. 학습 데이터를 자동 생성하는 **데모 생성기**

인터페이스는 SmolVLA 와 똑같은 `select_action()` 이라서 나중에 그대로 갈아끼웁니다.


In [ ]:
from vlapipe.policy import ScriptedSortingPolicy
import imageio.v2 as imageio
from IPython.display import Image as IPyImage, display

env.reset(layout='messy', seed=3)
policy = ScriptedSortingPolicy(env)
frames, log = [], []
for st in RulePlanner(env).plan():
    policy.set_subtask(st); n = 0
    while not policy.done and n < 500:
        env.step(policy.select_action()); n += 1
        if n % 3 == 0: frames.append(env.render('frontview', 300, 400))
    ok = env.zone_of(st['object']) == st['target_zone']
    log.append({'물체': st['object'], '스텝': n, '성공': ok, '실패사유': policy.failure})
display(pd.DataFrame(log))
print('정리 완료:', env.is_tidy(), '| 남은:', env.misplaced_objects())
imageio.mimsave('outputs/section4.gif', frames[::2], fps=14, loop=0)
display(IPyImage(filename='outputs/section4.gif'))

---
## SECTION 5 | 엔드투엔드 실행

한 번에 돌리는 스크립트입니다. 실패한 물체는 다음 패스에서 다시 시도합니다.


In [ ]:
!python scripts/run_pipeline.py --layout messy --seed 11 --passes 2 --planner rule

In [ ]:
# VLM 플래너로도 돌려 보기 (계획은 VLM, 실행은 scripted)
!python scripts/run_pipeline.py --layout messy --seed 11 --passes 2 --planner vlm

---
## SECTION 6 | 데이터 — 어지르고 나서 뒤집기

정확한 위치에 **내려놓기**는 손으로 어렵지만, 구역에서 **집어서 아무 데나 놓기**는 쉽습니다.
그래서 정돈된 씬을 어지르는 쪽을 게임패드로 녹화하고, 시간 역순으로 뒤집어 정돈 데모로 씁니다.

**녹화는 윈도우 로컬에서** (`scripts/teleop_record.py`), 뒤집기와 학습은 여기 Colab 에서 합니다.

### 뒤집기의 수학
`target[t+1] = target[t] + a[t]` 이므로 위치·회전 델타는 부호만 뒤집으면 됩니다:
`a'[k][:6] = -a[T-1-k][:6]`

그리퍼는 델타가 아니라 절대 명령이라 부호 반전이 안 됩니다. 대신
"역재생 기준 다음 프레임에서 손가락이 닫혀 있어야 하는가"로 다시 만듭니다.
그러면 원본의 *놓기* 순간이 역재생의 *잡기* 순간에 정확히 대응됩니다.


In [ ]:
!python scripts/build_dataset.py

In [ ]:
# 역재생이 제대로 됐는지 눈으로 확인
from vlapipe.episode import Episode
import glob
revs = sorted(glob.glob('data/episodes_reversed/*.hdf5'))
if revs:
    ep = Episode(revs[0])
    print(ep.path.name, '|', len(ep), '프레임 | task:', ep.task)
    idx = np.linspace(0, len(ep) - 1, 6).astype(int)
    fig, ax = plt.subplots(1, 6, figsize=(18, 3))
    for a, i in zip(ax, idx):
        a.imshow(ep.obs['agentview_image'][i]); a.axis('off'); a.set_title(f't={i}', fontsize=8)
    plt.suptitle('역재생된 정돈 데모 (왼쪽=시작, 오른쪽=끝)'); plt.tight_layout(); plt.show()
    imageio.mimsave('outputs/reversed.gif', list(ep.obs['agentview_image'][::3]), fps=12, loop=0)
    display(IPyImage(filename='outputs/reversed.gif'))
else:
    print('아직 녹화 데이터가 없습니다. 윈도우에서 scripts/teleop_record.py 를 먼저 실행하세요.')

---
## SECTION 7 | LeRobot 변환 & SmolVLA 파인튜닝

> 이 절은 **데이터를 모은 뒤**에 실행합니다. 에피소드 20~50개는 있어야 의미가 있습니다.


In [ ]:
# 7-1. LeRobot 설치 (시간이 좀 걸립니다)
!pip install -q 'lerobot[smolvla] @ git+https://github.com/huggingface/lerobot.git'

In [ ]:
# 7-2. LeRobot 데이터셋으로 변환
!python scripts/build_dataset.py --lerobot --repo-id dshs2026/vla_sorting

In [ ]:
# 7-3. SmolVLA 파인튜닝
#   base 모델 lerobot/smolvla_base 에서 출발합니다.
#   steps 는 처음엔 작게(2000) 잡고 학습곡선을 보며 늘리세요.
!python -m lerobot.scripts.train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=dshs2026/vla_sorting \
  --dataset.root=data/lerobot \
  --batch_size=8 --steps=2000 \
  --output_dir=outputs/smolvla_sorting \
  --job_name=vla_sorting --wandb.enable=false

In [ ]:
# 7-4. 학습된 정책으로 파이프라인 재실행
!python scripts/run_pipeline.py --policy smolvla \
  --checkpoint outputs/smolvla_sorting/checkpoints/last/pretrained_model \
  --layout messy --seed 11

---
## SECTION 8 | 미니과제

### 필수
1. **텔레옵 녹화 5개** — 정돈된 씬을 어지르는 장면. `build_dataset.py` 검증을 통과할 것
   (던지거나 떨어뜨린 구간이 있으면 자동 제외됩니다).
2. **프롬프트 배틀** — 팀원 3명이 서로 다른 프롬프트로 SECTION 3 채점 루프를 돌리고,
   5개 씬 평균 recall/precision 비교표를 만들 것. 가장 좋은 프롬프트를 고른 이유 1줄.
3. **실패 분석** — SECTION 4 의 실패 사유(`grasp_failed`, `dropped_during_carry`,
   `stalled_in_*`)를 물체 종류별로 집계하고, 왜 그 물체가 어려운지 2~3문장.

### 생각할 질문
- 지금 `agentview` 는 천장에서 내려다봅니다. 실제 로봇으로 옮길 때 이 시점을 못 쓴다면?
- scripted teacher 는 물체 좌표를 **직접 읽습니다**. SmolVLA 는 이미지만 봅니다.
  teacher 가 만든 데모로 학습한 정책이 teacher 만큼 잘할 수 있을까요? 무엇이 부족할까요?
- 역재생으로 만든 데모는 사람이 직접 정돈한 데모와 무엇이 다를까요?
  (힌트: 속도 분포, 접근 각도, 실패 후 재시도의 유무)


In [ ]:
# 제출용 저장
import json, shutil, datetime
TEAM = Path('/content/drive/MyDrive/VLA_Team_2026') if IN_COLAB else PKG.parent
dst = TEAM / 'results' / f'session12_{datetime.datetime.now():%m%d_%H%M}'
dst.mkdir(parents=True, exist_ok=True)
for p in Path('outputs').glob('**/*'):
    if p.is_file() and p.suffix in {'.gif', '.png', '.json'}:
        shutil.copy(p, dst / f'{p.parent.name}_{p.name}')
print('저장:', dst)
print(sorted(x.name for x in dst.iterdir()))